# Phase 5: ML Model Training
## Train and evaluate depression detection models

**Models:**
1. Random Forest Classifier (interpretable, feature importance)
2. Support Vector Machine (RBF kernel, baseline comparison)

**Evaluation:**
- 5-fold Stratified Cross-Validation
- Metrics: Accuracy, Precision, Recall, F1-score, ROC-AUC
- Confusion matrices for both models

In [1]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Load Feature Matrix

In [2]:
# Load features
features_path = Path('../data/features.csv')
features_df = pd.read_csv(features_path)

# Separate features and labels
X = features_df.drop('label', axis=1)
y = features_df['label']

print(f"✓ Features loaded: {X.shape}")
print(f"✓ Labels shape: {y.shape}")
print(f"✓ Class distribution: Healthy={np.sum(y==0)}, Depressed={np.sum(y==1)}")

✓ Features loaded: (30, 1116)
✓ Labels shape: (30,)
✓ Class distribution: Healthy=15, Depressed=15


## Model Training with Cross-Validation

In [3]:
# Define cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize models
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,  # For ROC-AUC
    random_state=42
)

print("✓ Models initialized")
print(f"  - Random Forest: n_estimators=100, max_depth=15")
print(f"  - SVM: kernel=rbf, C=1.0")

✓ Models initialized
  - Random Forest: n_estimators=100, max_depth=15
  - SVM: kernel=rbf, C=1.0


In [4]:
# Scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Train Random Forest with cross-validation
print("\nTraining Random Forest...")
rf_results = cross_validate(
    rf_model, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1
)

# Train SVM with cross-validation
print("Training SVM...")
svm_results = cross_validate(
    svm_model, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1
)

print("\n✓ Cross-validation complete!")


Training Random Forest...
Training SVM...

✓ Cross-validation complete!


In [5]:
# Display results
print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS (5-Fold Stratified)")
print("="*80)

# Random Forest Results
print("\nRANDOM FOREST:")
print("-" * 80)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = rf_results[f'test_{metric}']
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"  {metric.upper():<15} Mean: {mean_score:.4f} ± {std_score:.4f} | Folds: {scores}")

# SVM Results
print("\nSUPPORT VECTOR MACHINE (RBF):")
print("-" * 80)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = svm_results[f'test_{metric}']
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"  {metric.upper():<15} Mean: {mean_score:.4f} ± {std_score:.4f} | Folds: {scores}")

print("\n" + "="*80)


CROSS-VALIDATION RESULTS (5-Fold Stratified)

RANDOM FOREST:
--------------------------------------------------------------------------------
  ACCURACY        Mean: 0.5000 ± 0.1054 | Folds: [0.66666667 0.33333333 0.5        0.5        0.5       ]
  PRECISION       Mean: 0.4333 ± 0.2261 | Folds: [0.66666667 0.         0.5        0.5        0.5       ]
  RECALL          Mean: 0.4667 ± 0.2667 | Folds: [0.66666667 0.         0.66666667 0.33333333 0.66666667]
  F1              Mean: 0.4419 ± 0.2371 | Folds: [0.66666667 0.         0.57142857 0.4        0.57142857]
  ROC_AUC         Mean: 0.5333 ± 0.1515 | Folds: [0.55555556 0.38888889 0.33333333 0.72222222 0.66666667]

SUPPORT VECTOR MACHINE (RBF):
--------------------------------------------------------------------------------
  ACCURACY        Mean: 0.3667 ± 0.1247 | Folds: [0.16666667 0.33333333 0.5        0.33333333 0.5       ]
  PRECISION       Mean: 0.2800 ± 0.2315 | Folds: [0.  0.  0.5 0.4 0.5]
  RECALL          Mean: 0.3333 ± 0.298

## Model Comparison

In [6]:
# Create comparison dataframe
comparison_data = []

for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    rf_mean = np.mean(rf_results[f'test_{metric}'])
    rf_std = np.std(rf_results[f'test_{metric}'])
    svm_mean = np.mean(svm_results[f'test_{metric}'])
    svm_std = np.std(svm_results[f'test_{metric}'])
    
    comparison_data.append({
        'Metric': metric.upper(),
        'RF_Mean': rf_mean,
        'RF_Std': rf_std,
        'SVM_Mean': svm_mean,
        'SVM_Std': svm_std,
        'Difference': rf_mean - svm_mean
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nMODEL COMPARISON:")
print(comparison_df.to_string(index=False))

# Determine best model
rf_accuracy = np.mean(rf_results['test_accuracy'])
svm_accuracy = np.mean(svm_results['test_accuracy'])
best_model = "Random Forest" if rf_accuracy > svm_accuracy else "SVM"
print(f"\n✓ Best model: {best_model}")


MODEL COMPARISON:
   Metric  RF_Mean   RF_Std  SVM_Mean  SVM_Std  Difference
 ACCURACY 0.500000 0.105409  0.366667 0.124722    0.133333
PRECISION 0.433333 0.226078  0.280000 0.231517    0.153333
   RECALL 0.466667 0.266667  0.333333 0.298142    0.133333
       F1 0.441905 0.237110  0.294286 0.246378    0.147619
  ROC_AUC 0.533333 0.151535  0.266667 0.259153    0.266667

✓ Best model: Random Forest


## Get Predictions for Confusion Matrices

In [7]:
# Get cross-validated predictions
rf_pred = cross_val_predict(rf_model, X, y, cv=cv)
svm_pred = cross_val_predict(svm_model, X, y, cv=cv)

# Compute confusion matrices
rf_cm = confusion_matrix(y, rf_pred)
svm_cm = confusion_matrix(y, svm_pred)

print("✓ Cross-validated predictions obtained")
print(f"\nRandom Forest Confusion Matrix:")
print(rf_cm)
print(f"\nSVM Confusion Matrix:")
print(svm_cm)

✓ Cross-validated predictions obtained

Random Forest Confusion Matrix:
[[8 7]
 [8 7]]

SVM Confusion Matrix:
[[ 6  9]
 [10  5]]


## Train Final Models on Full Dataset

In [8]:
# Train final models on full dataset (for feature importance, deployment)
print("Training final models on full dataset...\n")

rf_final = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
print("✓ Random Forest trained on full data")

svm_final = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_final.fit(X, y)
print("✓ SVM trained on full data")

Training final models on full dataset...

✓ Random Forest trained on full data
✓ SVM trained on full data


## Save Models

In [9]:
# Save models
models_path = Path('../results/models')
models_path.mkdir(parents=True, exist_ok=True)

with open(models_path / 'rf_model.pkl', 'wb') as f:
    pickle.dump(rf_final, f)

with open(models_path / 'svm_model.pkl', 'wb') as f:
    pickle.dump(svm_final, f)

print(f"✓ Models saved to {models_path}")

✓ Models saved to ..\results\models


## Save Results Dictionary

In [10]:
# Save comprehensive results for analysis notebook
results_dict = {
    'rf_results': rf_results,
    'svm_results': svm_results,
    'rf_predictions': rf_pred,
    'svm_predictions': svm_pred,
    'rf_cm': rf_cm,
    'svm_cm': svm_cm,
    'feature_names': X.columns.tolist(),
    'rf_feature_importance': rf_final.feature_importances_,
    'labels': y.values
}

with open(models_path / 'results.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

print(f"✓ Results dictionary saved")
print(f"\nReady for Phase 6: Visualization & Analysis")

✓ Results dictionary saved

Ready for Phase 6: Visualization & Analysis
